# 08 - Unified Multi-View Dataset Builder

Builds the real unified-schema table (schema described in the root `README.md` / `data/README.md`) for every dataset available locally: State Farm, SAM-DD, and the Low-Light Driver Distraction Dataset. AUC V2 and 100-Driver are requested but not yet granted, so their adapters are written but won't run until the data exists.

SAM-DD has no label file, so its class mapping comes from my own visual audit (2026-08-04) — confidence recorded per class in `notes`, two classes not in the 10-class taxonomy excluded. Same caveat applies to Low-Light DD (Adapter 5): a real unresolved hand-side ambiguity on 4 of its 10 classes and a mixed day/night finding.

No training here, just reshaping existing metadata into the shared schema.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
SPLITS_DIR = PROJECT_ROOT / "data" / "splits"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABLES_DIR = PROJECT_ROOT / "results" / "tables"

UNIFIED_COLUMNS = [
    "image_path", "label_id", "label_name", "subject_id", "dataset_name",
    "camera_view", "vehicle_id", "lighting", "modality", "split",
    "is_synthetic", "notes",
]

# Fixed 10-class label space shared by every dataset (see README.md / data/README.md)
LABEL_NAMES = [
    "safe_driving", "texting_right", "phone_right", "texting_left", "phone_left",
    "adjusting_radio", "drinking", "reaching_behind", "hair_or_makeup", "talking_to_passenger",
]
CLASSCODE_TO_LABEL_ID = {f"c{i}": i for i in range(10)}

## Adapter 1: State Farm -> unified schema

Single fixed side-view camera, RGB, daytime. Reuses the existing subject-wise splits from notebook 02 as-is.

In [2]:
def build_state_farm_unified(split_name, split_csv_path):
    df = pd.read_csv(split_csv_path)

    unified = pd.DataFrame({
        "image_path": df["filepath"],
        "label_id": df["classname"].map(CLASSCODE_TO_LABEL_ID),
        "label_name": df["classname"].map(CLASSCODE_TO_LABEL_ID).map(lambda i: LABEL_NAMES[i]),
        "subject_id": "statefarm_" + df["subject"].astype(str),
        "dataset_name": "state_farm",
        "camera_view": "side",
        "vehicle_id": "unknown",
        "lighting": "day",
        "modality": "rgb",
        "split": split_name,
        "is_synthetic": False,
        "notes": "",
    })
    return unified[UNIFIED_COLUMNS]

state_farm_parts = [
    build_state_farm_unified("train", SPLITS_DIR / "state_farm_train_subject.csv"),
    build_state_farm_unified("val", SPLITS_DIR / "state_farm_val_subject.csv"),
    build_state_farm_unified("test", SPLITS_DIR / "state_farm_test_subject.csv"),
]
state_farm_unified = pd.concat(state_farm_parts, ignore_index=True)

print("State Farm unified rows:", len(state_farm_unified))
state_farm_unified.head()

State Farm unified rows: 22424


,image_path,label_id,label_name,subject_id,dataset_name,camera_view,vehicle_id,lighting,modality,split,is_synthetic,notes
0,D:\MSC_PROJECT\state-farm-distracted-driver-de...,0,safe_driving,statefarm_p014,state_farm,side,unknown,day,rgb,train,False,
1,D:\MSC_PROJECT\state-farm-distracted-driver-de...,0,safe_driving,statefarm_p014,state_farm,side,unknown,day,rgb,train,False,
2,D:\MSC_PROJECT\state-farm-distracted-driver-de...,0,safe_driving,statefarm_p014,state_farm,side,unknown,day,rgb,train,False,
3,D:\MSC_PROJECT\state-farm-distracted-driver-de...,0,safe_driving,statefarm_p014,state_farm,side,unknown,day,rgb,train,False,
4,D:\MSC_PROJECT\state-farm-distracted-driver-de...,0,safe_driving,statefarm_p014,state_farm,side,unknown,day,rgb,train,False,


### Sanity checks

Check nothing was lost or miscounted against the original splits, and that no label mapped to nothing.

In [3]:
assert state_farm_unified["label_id"].isna().sum() == 0, "Found unmapped classes - stop and check."
assert len(state_farm_unified) == sum(len(p) for p in state_farm_parts), "Row count mismatch."

# No subject should appear in more than one split (this must still hold - it's just being re-verified here)
subject_splits = state_farm_unified.groupby("subject_id")["split"].nunique()
leaking_subjects = subject_splits[subject_splits > 1]
assert len(leaking_subjects) == 0, f"Subject leakage detected: {leaking_subjects.index.tolist()}"

print("All checks passed: no unmapped labels, no row loss, no subject leakage across splits.")
display(state_farm_unified["split"].value_counts())
display(state_farm_unified["label_name"].value_counts())

All checks passed: no unmapped labels, no row loss, no subject leakage across splits.


split
train    15899
val       3439
test      3086
Name: count, dtype: int64

label_name
safe_driving            2489
texting_left            2346
phone_left              2326
drinking                2325
phone_right             2317
adjusting_radio         2312
texting_right           2267
talking_to_passenger    2129
reaching_behind         2002
hair_or_makeup          1911
Name: count, dtype: int64

## Adapter 2: SAM-DD -> unified schema

Adds a front view plus a comparable side view, recorded on an indoor simulator rig (not a real car — flagged via `vehicle_id`). No label file for the 0-9 folders, so the mapping is from a visual audit, not an authoritative source (see intro). Classes 7 (adjusting glasses) and 9 (fatigue/head-drop) aren't in the 10-class taxonomy and are excluded.

Split subject-wise myself (70/15/15, seeded) rather than trusting the dataset's own `Tester`/`Val` folder naming — 28 `Tester` + 14 `Val` = 42, exactly the paper's participant count, so these look like two recording batches rather than an actual split.

In [4]:
SAM_DD_ROOT = PROJECT_ROOT / "Sam_DD-Dataset" / "SAM-DD(RGB)" / "SAM-DD(RGB)"

# (label_id, label_name, confidence) - confidence from visual audit, 2026-08-04.
# label_id = -1 means "not in the fixed 10-class taxonomy - excluded".
SAM_DD_CLASS_MAP = {
    "0": (0, "safe_driving", "high"),
    "1": (6, "drinking", "high"),
    "2": (2, "phone_right", "medium"),
    "3": (4, "phone_left", "low - hand side unconfirmed, arbitrarily paired with class 2"),
    "4": (1, "texting_right", "high"),
    "5": (3, "texting_left", "low - hand side unconfirmed, arbitrarily paired with class 4"),
    "6": (8, "hair_or_makeup", "medium"),
    "7": (-1, "unmapped_adjusting_glasses", "excluded - not in fixed taxonomy"),
    "8": (7, "reaching_behind", "low - visually ambiguous, best-guess"),
    "9": (-1, "unmapped_fatigue_head_drop", "excluded - not in fixed taxonomy"),
}

VIEW_FOLDER_TO_CAMERA_VIEW = {"front_RGB": "front", "side_RGB": "side"}


def build_sam_dd_unified(seed=42):
    subject_dirs = sorted(d for d in SAM_DD_ROOT.iterdir() if d.is_dir())

    rows = []
    for subject_dir in subject_dirs:
        subject_id = "samdd_" + subject_dir.name.lower()
        for class_dir in sorted(subject_dir.iterdir()):
            if not class_dir.is_dir() or class_dir.name not in SAM_DD_CLASS_MAP:
                continue
            label_id, label_name, confidence = SAM_DD_CLASS_MAP[class_dir.name]

            for view_folder, camera_view in VIEW_FOLDER_TO_CAMERA_VIEW.items():
                view_dir = class_dir / view_folder
                if not view_dir.exists():
                    continue
                for img_path in view_dir.glob("*.jpg"):
                    rows.append({
                        "image_path": str(img_path),
                        "label_id": label_id,
                        "label_name": label_name,
                        "subject_id": subject_id,
                        "dataset_name": "sam_dd",
                        "camera_view": camera_view,
                        "vehicle_id": "simulator_rig",
                        "lighting": "indoor_artificial",
                        "modality": "rgb",
                        "split": None,
                        "is_synthetic": False,
                        "notes": f"class_mapping_confidence={confidence}",
                    })

    df = pd.DataFrame(rows)

    unmapped_count = (df["label_id"] == -1).sum()
    print(f"SAM-DD: excluding {unmapped_count} images with no match in the fixed 10-class taxonomy "
          f"(adjusting glasses, fatigue/head-drop)")
    df = df[df["label_id"] != -1].reset_index(drop=True)

    # Subject-wise split, done ourselves (not trusting Tester/Val folder naming - see markdown above)
    subjects = sorted(df["subject_id"].unique())
    rng = np.random.default_rng(seed)
    shuffled_subjects = list(subjects)
    rng.shuffle(shuffled_subjects)

    n = len(shuffled_subjects)
    n_train = int(n * 0.7)
    n_val = int(n * 0.15)
    train_subjects = set(shuffled_subjects[:n_train])
    val_subjects = set(shuffled_subjects[n_train:n_train + n_val])

    def assign_split(subject_id):
        if subject_id in train_subjects:
            return "train"
        elif subject_id in val_subjects:
            return "val"
        return "test"

    df["split"] = df["subject_id"].map(assign_split)

    return df[UNIFIED_COLUMNS]


sam_dd_unified = build_sam_dd_unified()

print("SAM-DD unified rows:", len(sam_dd_unified))
print("SAM-DD unique subjects:", sam_dd_unified["subject_id"].nunique())
display(sam_dd_unified["camera_view"].value_counts())
display(sam_dd_unified["label_name"].value_counts())
sam_dd_unified.head()

SAM-DD: excluding 6808 images with no match in the fixed 10-class taxonomy (adjusting glasses, fatigue/head-drop)
SAM-DD unified rows: 95542
SAM-DD unique subjects: 42


camera_view
front    47771
side     47771
Name: count, dtype: int64

label_name
safe_driving       56528
drinking            6918
texting_left        6326
texting_right       6148
phone_left          6096
phone_right         5862
hair_or_makeup      4122
reaching_behind     3542
Name: count, dtype: int64

,image_path,label_id,label_name,subject_id,dataset_name,camera_view,vehicle_id,lighting,modality,split,is_synthetic,notes
0,D:\MSC_PROJECT\Sam_DD-Dataset\SAM-DD(RGB)\SAM-...,0,safe_driving,samdd_tester1,sam_dd,front,simulator_rig,indoor_artificial,rgb,val,False,class_mapping_confidence=high
1,D:\MSC_PROJECT\Sam_DD-Dataset\SAM-DD(RGB)\SAM-...,0,safe_driving,samdd_tester1,sam_dd,front,simulator_rig,indoor_artificial,rgb,val,False,class_mapping_confidence=high
2,D:\MSC_PROJECT\Sam_DD-Dataset\SAM-DD(RGB)\SAM-...,0,safe_driving,samdd_tester1,sam_dd,front,simulator_rig,indoor_artificial,rgb,val,False,class_mapping_confidence=high
3,D:\MSC_PROJECT\Sam_DD-Dataset\SAM-DD(RGB)\SAM-...,0,safe_driving,samdd_tester1,sam_dd,front,simulator_rig,indoor_artificial,rgb,val,False,class_mapping_confidence=high
4,D:\MSC_PROJECT\Sam_DD-Dataset\SAM-DD(RGB)\SAM-...,0,safe_driving,samdd_tester1,sam_dd,front,simulator_rig,indoor_artificial,rgb,val,False,class_mapping_confidence=high


In [5]:
assert sam_dd_unified["label_id"].isna().sum() == 0, "Found unmapped classes - stop and check."
assert (sam_dd_unified["label_id"] == -1).sum() == 0, "Unmapped (-1) rows leaked through - stop and check."

subject_splits = sam_dd_unified.groupby("subject_id")["split"].nunique()
leaking_subjects = subject_splits[subject_splits > 1]
assert len(leaking_subjects) == 0, f"Subject leakage detected: {leaking_subjects.index.tolist()}"

# Confirm front and side views exist for the same subjects (paired cross-view data)
subjects_with_front = set(sam_dd_unified[sam_dd_unified["camera_view"] == "front"]["subject_id"])
subjects_with_side = set(sam_dd_unified[sam_dd_unified["camera_view"] == "side"]["subject_id"])
print("Subjects with front view:", len(subjects_with_front))
print("Subjects with side view:", len(subjects_with_side))
print("Subjects with both views:", len(subjects_with_front & subjects_with_side))

print("\nAll SAM-DD checks passed: no unmapped labels, no subject leakage across splits.")
display(sam_dd_unified["split"].value_counts())
display(sam_dd_unified.groupby(["camera_view", "split"]).size())

Subjects with front view: 42
Subjects with side view: 42
Subjects with both views: 42

All SAM-DD checks passed: no unmapped labels, no subject leakage across splits.


split
train    64312
val      16324
test     14906
Name: count, dtype: int64

camera_view  split
front        test      7453
             train    32156
             val       8162
side         test      7453
             train    32156
             val       8162
dtype: int64

## Adapter 3: AUC Distracted Driver Dataset V2 (not runnable yet)

Access requested (see the "Requested, not yet integrated" section of `references/dataset_sources.md`), not granted yet. Written ahead of time so I just set the path once the data exists — don't run until then.

Camera view is `front` (phone on passenger seat facing driver), different from State Farm's `side`. Per the driver's-own-left/right rule, need to check AUC V2's own docs before trusting `phone_left`/`phone_right`, since a front camera mirrors the image.

In [ ]:
AUC_V2_RAW_DIR = PROJECT_ROOT / "data" / "raw" / "auc_v2"

def build_auc_v2_unified():
    raise NotImplementedError(
        "AUC V2 has not been downloaded yet - access request is pending (see references/dataset_sources.md). "
        "Once the licence is granted and the data is placed under data/raw/auc_v2/, implement this adapter: "
        "read AUC V2's own metadata/folder structure, map its class names to LABEL_NAMES above "
        "(verify left/right against AUC V2's own documentation - do not assume it matches State Farm's convention), "
        "prefix subject_id with 'aucv2_', set camera_view='front', modality='rgb', lighting='unknown' unless documented, "
        "and build train/val/test split with subject-wise splitting (reuse the logic from notebook 02)."
    )

# auc_v2_unified = build_auc_v2_unified()  # uncomment once data is available

## Adapter 4: 100-Driver (not runnable yet)

Access requested (see the "Requested, not yet integrated" section of `references/dataset_sources.md`), not granted yet. Has 4 camera views (`front_left`, `front`, `front_right`, `side_right`) and 2 lighting/modality conditions (day/rgb, night/nir) — both read from folder structure once available, not a fixed value like State Farm's.

In [ ]:
HUNDRED_DRIVER_RAW_DIR = PROJECT_ROOT / "data" / "raw" / "100_driver"

def build_100_driver_unified():
    raise NotImplementedError(
        "100-Driver has not been downloaded yet - access request is pending (see references/dataset_sources.md). "
        "Once granted and placed under data/raw/100_driver/, implement this adapter: map its 21 distraction "
        "sub-types down to the 10 shared classes (document every merge decision - see README.md / data/README.md), "
        "read camera_view per image from its folder structure (front_left/front/front_right/side_right), "
        "set lighting='day'+modality='rgb' or lighting='night'+modality='nir' per source folder, "
        "prefix subject_id with '100driver_', and build a subject-wise split."
    )

# hundred_driver_unified = build_100_driver_unified()  # uncomment once data is available

## Adapter 5: Low-Light Driver Distraction Dataset -> unified schema

The "Novel Driver Distraction Dataset" (Saad, Khalil and Abbas, ICCES 2020; Mendeley DOI `10.17632/ykmr99nrsg.2`), extracted at `Novel_Driver_Distraction-Dataset/extracted/dataset/`. Single fixed side-view camera like State Farm, 70 real drivers, `c0`-`c9` folders, 52,350 real JPEGs, plus its own `Data.csv` (`index,driver,class,path,type`). The `type` column is the dataset's own split — ignored here in favour of this project's subject-wise splitting, same as the other adapters.

Originally scoped for notebook 06 only (single-view), added here so notebook 06 can pull from the same shared schema.

Two findings from an audit on 2026-08-17:

1. Mixed day/night, not uniformly night. A brightness check (mean grayscale luminance per driver, cross-checked visually on 18+ drivers) found only 9 of 70 drivers are genuinely night (`driver00, driver01, driver19-driver25` — dark, violet IR-illuminator visible). The other 61 were recorded in daylight through the same NoIR-style camera, giving the same false-colour look. Night measures ~12-28 luminance, day ~60-155, clean separation, consistent per driver across all 10 classes. `lighting` is assigned per-driver from this measurement, not a blanket value — notebook 06's night-vs-day comparison only uses the 9 confirmed-night drivers, not the originally assumed 52,350 night images.
2. Class-mapping audit (60+ images, cross-checked independently): `c0`, `c5`, `c6` map cleanly (high confidence), `c8`/`c9` medium-high, `c7` medium (reach is clear where present, but some clips show only ordinary driving posture). `c1`/`c3` (texting) and `c2`/`c4` (phone-to-ear) are each a clear action pair with an unresolvable hand-side split — every sample shows the same hand/side for both members, no way to tell left vs right from this single side-profile angle. Same issue as SAM-DD (Adapter 2): kept mapped rather than excluded, but marked low confidence on the left/right assignment, using State Farm's digit convention as an unverified default.

No classes excluded outright here (unlike SAM-DD's 2), but 5 of 10 classes carry weaker confidence (the 4 hand-side classes plus `c7`).

In [ ]:
from PIL import Image

LOW_LIGHT_ROOT = PROJECT_ROOT / "Novel_Driver_Distraction-Dataset" / "extracted" / "dataset"
LOW_LIGHT_CSV = LOW_LIGHT_ROOT / "Data.csv"
LOW_LIGHT_DATA_DIR = LOW_LIGHT_ROOT / "Data"

# (label_id, label_name, confidence) - confidence from visual audit, 2026-08-17. See the markdown
# cell above for the full reasoning. label_id = -1 would mean "excluded" (not needed here - every
# digit maps to something in the fixed taxonomy).
LOW_LIGHT_CLASS_MAP = {
    "c0": (0, "safe_driving", "high"),
    "c1": (1, "texting_right",
           "low - hand side (left/right) not distinguishable from this single camera angle; "
           "the action itself (phone held low near lap, looking down, other arm on wheel) is "
           "clearly texting and consistent across every driver checked. Paired with c3 following "
           "State Farm's digit convention - not independently verified"),
    "c2": (2, "phone_right",
           "low - phone-to-ear posture is unambiguous, but visually identical to c4 in every "
           "sample checked (same hand, same ear, no observed variation). Paired with c4 following "
           "State Farm's digit convention - not independently verified"),
    "c3": (3, "texting_left", "low - visually identical to c1 in every sample checked; see c1 note"),
    "c4": (4, "phone_left", "low - visually identical to c2 in every sample checked; see c2 note"),
    "c5": (5, "adjusting_radio", "high"),
    "c6": (6, "drinking", "high"),
    "c7": (7, "reaching_behind",
           "medium - unambiguous where present (clear torso-twist/arm-extension), but several "
           "sampled clips show only ordinary driving posture throughout, suggesting the reach is "
           "brief or weakly captured in some recordings"),
    "c8": (8, "hair_or_makeup",
           "medium-high - consistent hand-to-head/hair gesture across drivers, though sometimes "
           "looks more like touching glasses/temple or reaching toward the sun visor"),
    "c9": (9, "talking_to_passenger",
           "medium-high - driver turns toward camera/passenger side with visibly open mouth "
           "mid-speech in most samples checked; a minority of samples show weaker evidence"),
}


def classify_driver_lighting(driver_dir, luminance_threshold=40.0):
    """Classify a driver folder as 'day' or 'night' from the mean grayscale luminance of one
    representative frame (c0's first image). Verified against a manual visual audit of 18+
    drivers spanning both clusters: real night frames (violet/purple IR-illuminator light,
    genuinely dark scenes) measured ~12-28 on this scale; real day frames (bright, still through
    the same NoIR-style camera, giving the characteristic blown-highlight/pink-foliage look)
    measured ~60-155. The two clusters are separated by a wide gap, so a single threshold at 40
    is safe. Also checked that lighting is consistent across all 10 class folders within a given
    driver (it is, in every driver spot-checked this way) - so one frame per driver is enough."""
    c0_dir = driver_dir / "c0"
    files = sorted(c0_dir.glob("*.jpg"))
    if not files:
        return "unknown"
    img = Image.open(files[0]).convert("L")
    mean_lum = np.array(img, dtype=np.float32).mean()
    return "night" if mean_lum < luminance_threshold else "day"


def split_subject_pool(subject_list, seed):
    """70/15/15 subject-wise split of one pool of subject IDs. Returns (train_set, val_set) -
    anything not in either is test."""
    rng = np.random.default_rng(seed)
    shuffled = list(subject_list)
    rng.shuffle(shuffled)
    n = len(shuffled)
    n_train = int(n * 0.7)
    n_val = int(n * 0.15)
    return set(shuffled[:n_train]), set(shuffled[n_train:n_train + n_val])


def build_low_light_unified(seed=42):
    df = pd.read_csv(LOW_LIGHT_CSV)
    df["class_folder"] = "c" + df["class"].astype(str)

    driver_lighting = {}
    for driver_dir in sorted(LOW_LIGHT_DATA_DIR.iterdir()):
        if driver_dir.is_dir():
            driver_lighting[driver_dir.name] = classify_driver_lighting(driver_dir)

    night_drivers = sorted(d for d, l in driver_lighting.items() if l == "night")
    day_drivers = sorted(d for d, l in driver_lighting.items() if l == "day")
    print(f"Low-Light DD lighting audit: {len(night_drivers)} driver(s) genuinely night, "
          f"{len(day_drivers)} driver(s) actually daytime (same camera, different ambient light).")
    print("Night drivers:", night_drivers)

    rows = []
    for _, r in df.iterrows():
        cf = r["class_folder"]
        if cf not in LOW_LIGHT_CLASS_MAP:
            continue
        label_id, label_name, confidence = LOW_LIGHT_CLASS_MAP[cf]
        driver_folder = f"driver{int(r['driver']):02d}"
        rows.append({
            "image_path": str(LOW_LIGHT_ROOT / r["path"]),
            "label_id": label_id,
            "label_name": label_name,
            "subject_id": f"lowlightdd_{driver_folder}",
            "dataset_name": "low_light_dd",
            "camera_view": "side",
            "vehicle_id": "unknown",
            "lighting": driver_lighting.get(driver_folder, "unknown"),
            "modality": "nir",
            "split": None,
            "is_synthetic": False,
            "notes": f"class_mapping_confidence={confidence}",
        })
    ldf = pd.DataFrame(rows)

    unmapped_count = (ldf["label_id"] == -1).sum()
    if unmapped_count:
        print(f"Low-Light DD: excluding {unmapped_count} images with no confident match in the fixed 10-class taxonomy")
        ldf = ldf[ldf["label_id"] != -1].reset_index(drop=True)

    # Subject-wise AND lighting-aware split, own 70 drivers - the dataset's own `type` column
    # (train/validation/test) is intentionally ignored (see markdown cell above). Night and day
    # driver pools are split SEPARATELY (each 70/15/15) so both lighting conditions are
    # guaranteed to appear in every split. A first version of this cell split all 70 drivers
    # together in one pool and, purely by chance of the seed, put zero of the 9 night drivers in
    # the test split - caught by inspecting the per-lighting split breakdown in the next cell,
    # fixed here by splitting each lighting pool independently before combining.
    night_subject_ids = sorted(f"lowlightdd_{d}" for d in night_drivers)
    day_subject_ids = sorted(f"lowlightdd_{d}" for d in day_drivers)

    night_train, night_val = split_subject_pool(night_subject_ids, seed)
    day_train, day_val = split_subject_pool(day_subject_ids, seed)
    train_subjects = night_train | day_train
    val_subjects = night_val | day_val

    def assign_split(sid):
        if sid in train_subjects:
            return "train"
        elif sid in val_subjects:
            return "val"
        return "test"

    ldf["split"] = ldf["subject_id"].map(assign_split)
    return ldf[UNIFIED_COLUMNS]


low_light_unified = build_low_light_unified()

print("\nLow-Light DD unified rows:", len(low_light_unified))
print("Low-Light DD unique subjects:", low_light_unified["subject_id"].nunique())
display(low_light_unified["lighting"].value_counts())
display(low_light_unified["label_name"].value_counts())
low_light_unified.head()

In [9]:
assert low_light_unified["label_id"].isna().sum() == 0, "Found unmapped classes - stop and check."
assert (low_light_unified["label_id"] == -1).sum() == 0, "Unmapped (-1) rows leaked through - stop and check."

subject_splits = low_light_unified.groupby("subject_id")["split"].nunique()
leaking_subjects = subject_splits[subject_splits > 1]
assert len(leaking_subjects) == 0, f"Subject leakage detected: {leaking_subjects.index.tolist()}"

print("All Low-Light DD checks passed: no unmapped labels, no subject leakage across splits.")
display(low_light_unified["split"].value_counts())
display(low_light_unified.groupby(["lighting", "split"]).size())

# The night subset specifically - this is what notebook 06 actually trains/evaluates the
# low-light comparison on, so its size matters more than the full 52,350-row total.
night_only = low_light_unified[low_light_unified["lighting"] == "night"]
print(f"\nNight-only subset: {len(night_only)} images, {night_only['subject_id'].nunique()} drivers")
display(night_only.groupby("split")["subject_id"].nunique())
display(night_only["split"].value_counts())

All Low-Light DD checks passed: no unmapped labels, no subject leakage across splits.


split
train    35925
test      9000
val       7425
Name: count, dtype: int64

lighting  split
day       test      7500
          train    31425
          val       6675
night     test      1500
          train     4500
          val        750
dtype: int64


Night-only subset: 6750 images, 9 drivers


split
test     2
train    6
val      1
Name: subject_id, dtype: int64

split
train    4500
test     1500
val       750
Name: count, dtype: int64

## Save what's available today

State Farm, SAM-DD, and Low-Light DD are all real now, so all three get concatenated and saved. Will re-run and grow further once AUC V2 and/or 100-Driver are added.

In [10]:
combined_unified = pd.concat([state_farm_unified, sam_dd_unified, low_light_unified], ignore_index=True)

output_path = PROCESSED_DIR / "unified_multiview_metadata.csv"
combined_unified.to_csv(output_path, index=False)
print("Saved combined unified metadata (State Farm + SAM-DD + Low-Light DD) to:", output_path)
print("Total rows:", len(combined_unified))

summary = combined_unified.groupby(["dataset_name", "camera_view", "lighting", "split"]).size().reset_index(name="num_images")
display(summary)

summary_path = TABLES_DIR / "unified_multiview_metadata_summary.csv"
summary.to_csv(summary_path, index=False)
print("Saved summary table to:", summary_path)

Saved combined unified metadata (State Farm + SAM-DD + Low-Light DD) to: D:\MSC_PROJECT\data\processed\unified_multiview_metadata.csv
Total rows: 170316


,dataset_name,camera_view,lighting,split,num_images
0,low_light_dd,side,day,test,7500
1,low_light_dd,side,day,train,31425
2,low_light_dd,side,day,val,6675
3,low_light_dd,side,night,test,1500
4,low_light_dd,side,night,train,4500
5,low_light_dd,side,night,val,750
6,sam_dd,front,indoor_artificial,test,7453
7,sam_dd,front,indoor_artificial,train,32156
8,sam_dd,front,indoor_artificial,val,8162
9,sam_dd,side,indoor_artificial,test,7453


Saved summary table to: D:\MSC_PROJECT\results\tables\unified_multiview_metadata_summary.csv


## Summary

- Built the unified-schema table for State Farm, SAM-DD, and Low-Light DD (`data/processed/unified_multiview_metadata.csv`), reusing State Farm's splits and building fresh subject-wise splits for the other two.
- Low-Light DD's class mapping followed SAM-DD's precedent: visual audit, per-class confidence, no forced guesses. No classes excluded, but 4 of 10 (texting/phone-call left-vs-right) carry a low-confidence caveat on hand side — the action itself isn't in doubt.
- Low-Light DD turned out to be mixed day/night (9 of 70 drivers genuinely night) rather than uniformly night as assumed — `lighting` assigned per-driver from a measured brightness check.
- Wrote but didn't run the AUC V2 and 100-Driver adapters — once those arrive, building the combined table is just uncommenting two lines.
- Next: once a new dataset is approved and downloaded, implement its adapter, run sanity checks, re-run the save cell.